# 05 — Most diversified portfolio

`MostDiversified` (in `maplab.models`): the long-only maximum-diversification-ratio portfolio, scored by the rolling backtest against EW and 60/40. It is risk-based: only Σ enters, no forecast of μ. Hypotheses are pre-registered in §2; cross-model tests are in notebooks 08–09.

## §1 Diversification as the objective

The diversification ratio is weighted-average asset volatility over portfolio volatility: 1 for a single asset, rising as correlations fall. The most diversified portfolio (MDP) maximizes it, which makes it the max-Sharpe portfolio when every asset has the same Sharpe ratio, so, like GMV, it is mean-variance with an implicit μ, here proportional to σ.

Notebook 04's UUP corner in GMV survived covariance shrinkage. GMV rewards low volatility and low correlation; MDP's risk weights reward only low correlation, and volatility enters only when risk weights become capital ($w_i \propto x_i/\sigma_i$, §3). If the corner is a volatility effect, MDP removes it; if it is a correlation effect, MDP keeps it.

## Setup

Prices, log returns for estimation, simple returns for the backtest, BIL as $r_f$, and the panel the classes read.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import maplab as ml
from maplab import (
    Panel, apply_style, GMV, MaxSharpe, EqualWeight, FixedWeight, MostDiversified,
    backtest, summary, mdp_closed_form,
)
from maplab.models import _min_variance_long_only, _mdp_long_only, _tangency_long_only

import _nb05_helpers as h

In [ ]:
apply_style()

prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)        # for estimation (mu, Sigma)
simple_returns = ml.to_simple_returns(prices)  # for compounding the backtest
rf_daily = ml.load_rf_returns()["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": ml.load_rf_returns()})
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)

rdates = ml.rebalance_dates(log_returns.index)
rdates = rdates[rdates >= ml.first_eligible_rebalance(log_returns.index)]
assert len(rdates) == 208, f"expected 208 rebalance labels, got {len(rdates)}"

## §2 Pre-registered hypotheses

Registered before any backtest below was run; the frozen text is in `registrations/nb05.toml`. MDP(S) is registered; MDP(LW) and inverse volatility (IV) are sensitivities. The H1/H2 directions were fixed by a read-only snapshot at 2014-12-31, 2017-12-31 and 2022-12-31. Paired tests use the daily return difference, $t = \bar d / s_d \cdot \sqrt{n}$, plain OLS; a pass on the full window only is reported as "not robust", never "supported".

| | Hypothesis | Counts if |
|---|---|---|
| H1 | effN(MDP) ≤ effN(GMV) | on more than 50% of the 208 rebalances and more than 50% of the 40 test rebalances |
| H2 | UUP weight MDP ≥ GMV | on more than 50% of the 208 rebalances and more than 50% of the 40 test rebalances, and the UUP corner persists at 2022-12-31 |
| H3 | MDP − GMV: null | t within ±2 on full and test windows; β decomposition reported |
| H4 | MDP − EW < 0, driven by β | full-window t ≤ −2 and a negative test-window mean, with the β part larger in size than Δα and Δα's t within ±2 |
| H5 | MDP(LW) − MDP(S): null; MDP − IV Δα: null | t within ±2 (full and test for LW) |

## §3 The math

**General form.** With $\sigma = \sqrt{\mathrm{diag}\,\Sigma}$ and $D = \mathrm{diag}(\sigma)$,

$$DR(w) = \frac{w'\sigma}{\sqrt{w'\Sigma w}} .$$

DR is the Sharpe ratio with excess returns replaced by σ, so the unconstrained maximizer on the budget $\mathbf 1'w = 1$ is the tangency formula with μ := σ:

$$w_{MDP} = \frac{\Sigma^{-1}\sigma}{\mathbf 1'\Sigma^{-1}\sigma} .$$

The first-order condition is $\Sigma w = \sigma\,(w'\Sigma w)/(w'\sigma)$, i.e. $\rho(r_i, r_p) = 1/DR^*$ for every asset.

**Long-only problem the class solves.** Maximize $DR(w)$ s.t. $\mathbf 1'w = 1$, $w \ge 0$. Same convex reformulation as `MaxSharpe`: minimize $y'\Sigma y$ s.t. $\sigma'y = 1$, $y \ge 0$, then $w = y/\mathbf 1'y$. It is exact and never degenerate, since $\sigma > 0$. At the optimum, held assets have $\rho(r_i, r_p) = 1/DR^*$ and excluded assets $\rho(r_i, r_p) \ge 1/DR^*$.

**Risk weights.** With $x_i = w_i\sigma_i / w'\sigma$ and $y = Dw$, the problem becomes the long-only GMV on the correlation matrix $C = D^{-1}\Sigma D^{-1}$. Correlations alone set $x$, and $w_i \propto x_i/\sigma_i$. At the optimum each asset's share of portfolio variance equals its risk weight, $w_i(\Sigma w)_i / w'\Sigma w = x_i$ (held assets share the same $\rho(r_i, r_p)$); for long-only GMV it equals the capital weight. GMV on Σ prefers low volatility and low correlation; MDP keeps only the correlation preference.

**Special cases.** Constant correlation ⇒ inverse volatility (IV). Equal volatilities ⇒ GMV.

**Estimator.** $x$ depends on $C$ only. Rescaling an asset leaves the sample $C$ unchanged but not the Ledoit–Wolf one (target $mI$), so MDP(LW) differs from MDP(S) partly for a reason other than estimation noise.

In [ ]:
asof_2022 = pd.Timestamp("2022-12-31")
_, Sigma_2022 = GMV(cov_estimator=ml.sample_cov)._estimate(panel, asof_2022)
w_mdp_unc_2022 = mdp_closed_form(Sigma_2022)
gross_2022 = float(w_mdp_unc_2022.abs().sum())
print(f"unconstrained MDP gross leverage sum|w| at 2022-12-31 = {gross_2022:.4f}")

Unconstrained, the MDP at 2022-12-31 has gross exposure $\sum_i \lvert w_i \rvert$ = 1.60, so it shorts; the long-only constraint binds.

The long-only optimality condition at 2022-12-31: each asset's correlation with the MDP against $1/DR^*$, and the asset with the largest risk weight.

In [ ]:
w_kkt, _ = _mdp_long_only(Sigma_2022, "KKT", asof_2022)
S_22 = Sigma_2022.to_numpy()
sig_22 = np.sqrt(np.diag(S_22))
sp_22 = float(np.sqrt(w_kkt @ S_22 @ w_kkt))
rho_p = (S_22 @ w_kkt) / (sig_22 * sp_22)
dr_star = float(w_kkt @ sig_22) / sp_22
held = w_kkt > 1e-4
x_risk = w_kkt * sig_22 / float(w_kkt @ sig_22)
names_22 = list(Sigma_2022.columns)
rc_share = w_kkt * (S_22 @ w_kkt) / sp_22 ** 2

held_dev = float(np.max(np.abs(rho_p[held] - 1 / dr_star)))
excluded_margin = float(np.min(rho_p[~held] - 1 / dr_star))
share_dev = float(np.max(np.abs(rc_share - x_risk)))
print(f"DR* = {dr_star:.3f}, 1/DR* = {1 / dr_star:.4f}; {int(held.sum())} of {len(held)} held; "
      f"held assets' correlation with the MDP equals 1/DR* to {held_dev:.1e}, "
      f"excluded assets' is higher by at least {excluded_margin:.1e}")
print(f"variance share = risk weight to {share_dev:.1e}; "
      f"largest risk weight {names_22[int(np.argmax(x_risk))]} {x_risk.max():.4f}")

**Reading.** 6 of 13 assets are held; their correlations with the MDP equal $1/DR^*$ = 0.2656, and every excluded asset's correlation is higher. Each held asset's variance share equals its risk weight; UUP has the largest, x = 0.4167.

## §4 The class and the rolling backtest

`MostDiversified` estimates Σ with `_estimate` (defined in notebook 01 §3) and passes it to `_mdp_long_only` in `src/maplab/models.py`, which calls the solver shared with `MaxSharpe` with σ in place of excess returns.

### Backtests

Seven runs, harness as in notebook 01 §3. MDP(S) is registered; MDP(LW) and IV are sensitivities; GMV(S), MaxSharpe(S), EW and 60/40 are the reproduction gate, and EW and 60/40 are the benchmarks.

In [ ]:
strategies = {
    "GMV(S)": GMV(cov_estimator=ml.sample_cov),
    "MaxSharpe(S)": MaxSharpe(cov_estimator=ml.sample_cov),
    "EW": EqualWeight(),
    "60/40": FixedWeight({"SPY": 0.6, "IEF": 0.4}, name="60/40"),
    "MDP(S)": MostDiversified(cov_estimator=ml.sample_cov),
    "MDP(LW)": MostDiversified(cov_estimator=ml.ledoit_wolf_cov),
    "IV": ml.InverseVol(),
}

results = {}
for name, strat in strategies.items():
    net, wlog, diag = backtest(strat, simple_returns, panel)
    results[name] = {"net": net, "wlog": wlog, "diag": diag, "strat": strat}

### Reproduction gate

GMV(S), MaxSharpe(S), EW and 60/40 must match notebooks 01–04 to 2 dp in every window; otherwise the cell stops the notebook.

In [ ]:
rows = []
for name, r in results.items():
    net = r["net"]
    diag = r["diag"]
    turnover = diag["turnover_per_rebalance"]
    win_defs = {
        "full": (net.index >= net.index.min(), turnover.index >= turnover.index.min()),
        "train (<= split)": (net.index <= split_ts, turnover.index <= split_ts),
        "test (> split)": (net.index > split_ts, turnover.index > split_ts),
    }
    for window_name, (net_sel, t_sel) in win_defs.items():
        t_window = turnover.loc[t_sel]
        s = summary(net.loc[net_sel], t_window, rf=rf_daily)
        s["strategy"] = name
        s["window"] = window_name
        rows.append(s)

summary_table = pd.DataFrame(rows).set_index(["strategy", "window"])
summary_table = summary_table[["ann_return", "ann_vol", "sharpe", "max_dd", "calmar", "hit_rate", "ann_turnover"]]
summary_table["cost_drag"] = summary_table["ann_turnover"] * ml.COST_BPS / 1e4

In [ ]:
ml.inference.check_reproduction(summary_table, {"GMV(S)": "GMV_sample", "MaxSharpe(S)": "MaxSharpe_sample", "EW": "EW", "60/40": "SixtyForty_SPY_IEF"})

n_retry_s = len(results["MDP(S)"]["strat"].retry_dates)
n_retry_lw = len(results["MDP(LW)"]["strat"].retry_dates)
print("reproduction gate passed (GMV(S), MaxSharpe(S), EW, 60/40 vs notebooks 01–04, 2 dp); "
      f"MDP retries: S {n_retry_s}, LW {n_retry_lw}")

### Solver checks

At all 208 rebalances, under the sample and Ledoit–Wolf Σ: the §3 variance-share identity, the MDP's DR margin over GMV, EW and IV, and retries.

In [ ]:
mech_table = h.mechanism_checks(panel, rdates)

for _, row in mech_table.iterrows():
    assert row["max_identity_diff"] <= 1e-5, f"identity check failed: {row['max_identity_diff']}"
    assert row["min_dr_margin"] >= -1e-10, f"DR optimality failed: {row['min_dr_margin']}"

print("solver checks at 208 rebalances, sample and LW: "
      f"variance-share identity ≤ {mech_table['max_identity_diff'].max():.1e}; "
      f"DR above GMV, EW and IV by at least {np.floor(100 * mech_table['min_dr_margin'].min()) / 100:.2f}")

## §5 Snapshot: 2014-12-31, 2017-12-31, 2022-12-31

Sample Σ at the three dates that fixed the H1/H2 directions: weights at 2022-12-31, then DR, ex-ante volatility and effective N at all three, then UUP's GMV weight g, MDP risk weight r and MDP weight m.

In [ ]:
DATES = [pd.Timestamp(d) for d in ["2014-12-31", "2017-12-31", "2022-12-31"]]

weight_tables, snapshot_metrics, uup_table = h.snapshot_tables(panel, results, DATES)

weights_shown = weight_tables[asof_2022.date()].rename_axis(None)
display(ml.reporting.show_table(weights_shown, pct=list(weights_shown.columns), pct_dp=1))

metrics_shown = snapshot_metrics.rename(columns={
    "DR": "diversification ratio", "exante_vol": "ex-ante vol (%)", "effN": "effective N",
    "max_w": "largest weight (%)", "max_ticker": "largest holding"})
display(ml.reporting.show_table(metrics_shown, pct=["ex-ante vol (%)", "largest weight (%)"], pct_dp=2, dp=2))

uup_shown = uup_table.rename(columns={
    "g": "g (GMV weight)", "r": "r (MDP risk weight)", "m": "m (MDP weight)",
    "uup_vol": "vol (%)", "uup_rank": "vol rank", "uup_mean_corr": "mean corr"}).rename_axis(None)
display(pd.concat([ml.reporting.show_table(uup_shown[["g (GMV weight)", "r (MDP risk weight)", "m (MDP weight)"]], dp=4),
                   ml.reporting.show_table(uup_shown[["vol (%)"]], pct=["vol (%)"], pct_dp=2),
                   ml.reporting.show_table(uup_shown[["vol rank", "mean corr"]], dp=2, int_cols=["vol rank"])], axis=1))

At 2022-12-31 UUP is the largest weight in GMV (58.12%) and MDP (59.22%). Correlation alone makes it the largest MDP risk weight, r = 0.4167, with a mean correlation of −0.47 with the other 12 assets; as the lowest-volatility asset, its capital weight is higher still. m ≥ g at all 3 dates: removing GMV's volatility preference does not shrink the corner, so its source is correlation, and low volatility enlarges it.

### Snapshot map at 2022-12-31

MDP, GMV, MaxSharpe, IV, EW and 60/40 in (volatility, μ) space. μ is the 252-day sample mean, so read the volatility axis; MaxSharpe appears at this date only.

In [ ]:
mu_2022, Sigma_2022 = GMV(cov_estimator=ml.sample_cov)._estimate(panel, asof_2022)
Sigma_np = Sigma_2022.to_numpy()
n = Sigma_np.shape[0]

w_gmv_2022 = _min_variance_long_only(Sigma_2022)
w_mdp_2022, _ = _mdp_long_only(Sigma_2022, "C1", asof_2022)
w_iv_2022 = (1.0 / np.sqrt(np.diag(Sigma_np))) / np.sum(1.0 / np.sqrt(np.diag(Sigma_np)))
w_ew_2022 = np.full(n, 1.0 / n)
rf_ann_2022 = MaxSharpe()._rf_ann(panel, asof_2022)
excess_2022 = mu_2022 - rf_ann_2022
if excess_2022.max() > 0:
    w_msr_2022, _ = _tangency_long_only(excess_2022.to_numpy(), Sigma_np, "C1", asof_2022)
else:
    w_msr_2022 = _min_variance_long_only(Sigma_2022)

fig = ml.plotting.snapshot_map(
    Sigma_2022, mu_2022,
    {"MDP": w_mdp_2022, "GMV": w_gmv_2022, "MaxSharpe": w_msr_2022, "IV": w_iv_2022, "EqualWeight": w_ew_2022,
     "60/40": FixedWeight({"SPY": 0.6, "IEF": 0.4}, name="60/40")(panel, asof_2022).to_numpy()},
    title=f"Snapshot {asof_2022.date()}: MDP, GMV, MaxSharpe, IV, EW, 60/40",
    figsize=(10, 6), legend_outside=True,
    styles={"60/40": dict(marker="D", s=130, color=ml.plotting.METHOD_COLORS["60/40"])},
)
plt.show()

MDP and GMV sit together at the minimum-volatility end of the frontier (ex-ante volatility 3.50% and 3.32%); IV and EW sit inside the cloud at 9.11% and 11.04%. Apart from DBC at the far end, UUP is the asset closest to the frontier, with low volatility and a positive 252-day mean.

### Capital vs risk at 2022-12-31

Capital weight (dark) and share of portfolio variance (light) per asset; a negative share is a hedge. Table columns as in notebook 01 §4.5.

In [ ]:
weights_2022 = {name: pd.Series(w, index=Sigma_2022.columns) for name, w in
                [("MDP", w_mdp_2022), ("GMV", w_gmv_2022), ("IV", w_iv_2022), ("EW", w_ew_2022),
                 ("60/40", FixedWeight({"SPY": 0.6, "IEF": 0.4}, name="60/40")(panel, asof_2022).to_numpy())]}
div = ml.diagnostics.diversification_table(weights_2022, Sigma_2022).rename(columns={
    "w_UUP": "UUP weight (%)", "effN": "effective N", "n_uncorr": "uncorrelated bets",
    "max_RC_share": "largest risk share (%)", "DR": "diversification ratio", "exante_vol": "ex-ante vol (%)"})
display(ml.reporting.show_table(div, pct=["UUP weight (%)", "largest risk share (%)", "ex-ante vol (%)"], pct_dp=2, dp=2))

fig = ml.plotting.capital_vs_risk(weights_2022, Sigma_2022, title="Capital weight vs risk share, 2022-12-31 (sample Σ)")
plt.show()

**Reading.** MDP has the highest diversification ratio (3.765), as it must; 60/40 has 1.18. By the §3 identity, UUP's variance share is its capital weight in GMV (58.12%) and its risk weight in MDP (41.67%): MDP holds more UUP capital than GMV but puts less of its risk there, spread over 7.17 uncorrelated bets (GMV 7.57, EW 1.40, 60/40 1.76).

### Mechanism: volatility vs correlation at 2022-12-31

Each asset's volatility against its mean correlation with the other 12; marker area = MDP(S) weight, hollow = zero weight. Correlation sets the risk weight and $1/\sigma$ converts it to capital, so MDP concentrates where correlation and volatility are both low.

In [ ]:
fig = h.vol_corr_figure(panel, results, asof_2022)
plt.show()

UUP is the only asset with a negative mean correlation (−0.47) and has the lowest volatility, so both effects push its weight up (59.22%). The credit ETFs (LQD, HYG, EMB) have the highest mean correlations and are all excluded. SPY and EFA have similar mean correlations but only EFA is held, because the §3 condition is on each asset's correlation with the portfolio, not its average correlation.

## §6 Out-of-sample results

Walk-forward as in notebook 01 §5; the test window (2023+) is the cleaner read, and wealth paths and formal tests are in notebook 08.

In [ ]:
full = summary_table.xs("full", level="window")
train = summary_table.xs("train (<= split)", level="window")
test = summary_table.xs("test (> split)", level="window")
overview = pd.DataFrame({
    "Sharpe full": full["sharpe"], "Sharpe train": train["sharpe"], "Sharpe test": test["sharpe"],
    "return (%)": full["ann_return"], "vol (%)": full["ann_vol"],
    "max DD (%)": full["max_dd"], "turnover (%/yr)": full["ann_turnover"],
}).loc[["MDP(S)", "MDP(LW)", "IV", "GMV(S)", "MaxSharpe(S)", "EW", "60/40"]].rename_axis(None)
ml.reporting.show_table(overview, pct=["return (%)", "vol (%)", "max DD (%)", "turnover (%/yr)"],
                        pct_dp=1, dp=2, int_cols=["turnover (%/yr)"])

MDP(S) has a full-window Sharpe of 0.76 (0.70 train, 0.98 test) at 3.46% volatility (EW 7.87%, 60/40 10.11%): above GMV(S)'s 0.62, between EW's 0.74 and 60/40's 0.91. MDP(LW) (0.76) and IV (0.72) are within 0.05. Turnover is 52%/yr against GMV's 71%.

## §7 What drove the results

### 7.1 Weights by asset group

Capital by asset group over time for MDP(S), GMV(S) and IV; below, the mean over the 208 rebalances, with EW and 60/40 for reference (%).

In [ ]:
fig = ml.plotting.group_stackplot(results, ["MDP(S)", "GMV(S)", "IV"], split_ts=split_ts)
plt.show()

In [ ]:
grp_mean = {}
for name in ["MDP(S)", "GMV(S)", "IV", "EW", "60/40"]:
    g = ml.diagnostics.weights_by_group(results[name]["wlog"])
    assert len(g) == 208, (name, len(g))
    grp_mean[name] = g.mean()
ml.reporting.show_table((100 * pd.DataFrame(grp_mean)).rename_axis(None), dp=1)

On average MDP(S) holds 44.8% in fx (UUP) and 13.6% in equity, against GMV(S)'s 37.0% and 7.0%. MDP holds almost no inflation-linked or credit (0.8% and 1.8%), where GMV holds 15.7% and 12.9% and IV 14.2% and 31.4%.

### 7.2 Concentration (H1)

Effective N, UUP weight and trailing 12-rebalance one-way turnover for MDP(S) and GMV(S); EW dashed, 60/40 as a reference line.

In [ ]:
effN_gmv_ts = 1.0 / (results["GMV(S)"]["wlog"] ** 2).sum(axis=1)
effN_mdp_ts = 1.0 / (results["MDP(S)"]["wlog"] ** 2).sum(axis=1)

fig = ml.plotting.concentration_panel(
    results, ["MDP(S)", "GMV(S)", "60/40"], rows=("effN", "highlight", "turnover"), ref="EW", split_ts=split_ts,
    figsize=(9.5, 10.5), colors={"60/40": ml.plotting.METHOD_COLORS["60/40"]},
    suptitle="MDP(S) vs GMV(S), EW and 60/40 for reference: effective N, UUP weight, turnover",
)
plt.show()

In [ ]:
le_full = int((effN_mdp_ts <= effN_gmv_ts).sum())
n_full = len(effN_mdp_ts)
test_mask = effN_mdp_ts.index > split_ts
le_test = int((effN_mdp_ts.loc[test_mask] <= effN_gmv_ts.loc[test_mask]).sum())
n_test = int(test_mask.sum())
median_mdp = float(effN_mdp_ts.median())
median_gmv = float(effN_gmv_ts.median())
n_uup_top = int((results["MDP(S)"]["wlog"].idxmax(axis=1) == "UUP").sum())

print(f"H1: {le_full}/{n_full} ({100*le_full/n_full:.1f}%), test {le_test}/{n_test}")
print(f"median effN: MDP(S)={median_mdp:.2f}  GMV(S)={median_gmv:.2f}")
print(f"UUP is MDP(S)'s largest weight at {n_uup_top} of {n_full} rebalances")

**Reading.** effN(MDP) ≤ effN(GMV) at 146 of 208 rebalances and 34 of 40 in the test window; median effN 3.35 vs 3.79. UUP is MDP's largest weight at 189 of 208 rebalances, and MDP's turnover is mostly below GMV's.

### 7.3 UUP weight (H2)

In [ ]:
w_uup_gmv_ts = results["GMV(S)"]["wlog"]["UUP"]
w_uup_mdp_ts = results["MDP(S)"]["wlog"]["UUP"]

ge_full = int((w_uup_mdp_ts >= w_uup_gmv_ts).sum())
n_full2 = len(w_uup_mdp_ts)
test_mask2 = w_uup_mdp_ts.index > split_ts
ge_test = int((w_uup_mdp_ts.loc[test_mask2] >= w_uup_gmv_ts.loc[test_mask2]).sum())
n_test2 = int(test_mask2.sum())

uup_2022_gmv = float(w_uup_gmv_ts.loc[pd.Timestamp("2022-12-31")])
uup_2022_mdp = float(w_uup_mdp_ts.loc[pd.Timestamp("2022-12-31")])

print(f"H2: {ge_full}/{n_full2} ({100*ge_full/n_full2:.1f}%), test {ge_test}/{n_test2}")
print(f"2022-12-31: GMV(S) UUP={100 * uup_2022_gmv:.2f}%  MDP(S) UUP={100 * uup_2022_mdp:.2f}%")

MDP holds at least as much UUP as GMV at 187 of 208 rebalances (38 of 40 test); at 2022-12-31, 59.22% vs 58.12%. Removing the volatility preference does not remove the corner.

### 7.4 Paired differences and the β split (H3–H5)

For the four registered pairs: daily net-return differences ($t = \bar d / s_d \cdot \sqrt{n}$, plain OLS, no HAC), CAPM regressions on SPY's excess return, the regression of each difference (Δα with its own t), and the split of each full-window difference into Δα and a β part.

In [ ]:
pairs = [
    ("MDP(S)", "GMV(S)"),
    ("MDP(S)", "EW"),
    ("MDP(LW)", "MDP(S)"),
    ("MDP(S)", "IV"),
]

paired_table, paired_series = ml.inference.paired_table(results, pairs, split_ts, summary_table)
paired_shown = paired_table[["ann_mean_diff", "TE", "t", "cost_drag_diff"]].rename(columns={
    "ann_mean_diff": "mean diff (%/yr)", "TE": "TE (%/yr)", "cost_drag_diff": "cost-drag diff (%/yr)"})
ml.reporting.show_table(paired_shown, pct=["mean diff (%/yr)", "TE (%/yr)", "cost-drag diff (%/yr)"], pct_dp=2, dp=2)

MDP(S) − GMV(S) is +0.64%/yr (t 1.98) full and +1.10%/yr (t 1.68) test; MDP(S) − EW is −3.17%/yr (t −2.00) full and −3.20%/yr test. MDP(LW) − MDP(S) has t 1.31 full and 0.53 test. Cost-drag differences are at most 0.04%/yr, so costs change no sign.

In [ ]:
MKT_simple = (simple_returns["SPY"] - rf_daily).dropna()

capm_table = ml.inference.capm_table(results, ["MDP(S)", "GMV(S)", "EW", "60/40", "MDP(LW)", "IV"],
                                     MKT_simple, rf_daily)
capm_shown = capm_table.rename(columns={"alpha_ann": "α (%/yr)", "t_alpha": "t(α)", "beta": "β", "r2": "R²"}).rename_axis(None)
ml.reporting.show_table(capm_shown, pct=["α (%/yr)"], pct_dp=2, dp=2, int_cols=["n"])

In [ ]:
# Δα with its own t (the H4/H5 statistic), not α_a − α_b
diff_capm_table = ml.inference.diff_regression(paired_series, pairs, MKT_simple)
diff_shown = diff_capm_table.rename(columns={
    "alpha_diff_ann": "Δα (%/yr)", "t_alpha_diff": "t(Δα)", "beta_diff": "Δβ"}).rename_axis(None)
pd.concat([ml.reporting.show_table(diff_shown[["Δα (%/yr)", "t(Δα)"]], pct=["Δα (%/yr)"], pct_dp=2, dp=2),
           ml.reporting.show_table(diff_shown[["Δβ"]], dp=4),
           ml.reporting.show_table(diff_shown[["n"]], int_cols=["n"])], axis=1)

In [ ]:
decomp = ml.inference.beta_decomposition(capm_table, paired_table, diff_capm_table, pairs,
                                         results, rf_daily, MKT_simple)
assert all(abs(r["identity_check"]) < 1e-9 for r in decomp.values()), "decomposition does not reproduce the difference"

decomp_shown = pd.DataFrame({
    pair: {"Δα (%/yr)": r["d_alpha"], "t(Δα)": r["t_alpha_diff"], "Δβ": r["d_beta"],
           "β part (%/yr)": r["beta_part"], "difference (%/yr)": r["realized_diff_full"]}
    for pair, r in decomp.items()}).T.astype(float)
pd.concat([ml.reporting.show_table(decomp_shown[["Δα (%/yr)", "t(Δα)"]], pct=["Δα (%/yr)"], pct_dp=2, dp=2),
           ml.reporting.show_table(decomp_shown[["Δβ"]], dp=4),
           ml.reporting.show_table(decomp_shown[["β part (%/yr)", "difference (%/yr)"]],
                                   pct=["β part (%/yr)", "difference (%/yr)"], pct_dp=2)], axis=1)

**Reading.** β split (full window): MDP(S) − GMV(S) is Δα +0.35%/yr plus a β part of +0.29%/yr; MDP(S) − EW is a β part of −4.19%/yr against Δα +1.02%/yr (t 1.00), so its shortfall comes from lower β. MDP(S) − IV has Δα t 0.71, and MDP(S)'s own CAPM α has a plain-OLS t of 2.02; notebook 08 tests α with Newey–West errors and multiple-testing control.

Cumulative paired daily differences for the registered pairs. Top: pairs registered on the total difference. Bottom: β-adjusted (solid, d_t − β̂·mkt_t) and total (dashed), for the pairs whose registered or reported statistic is Δα or the β split.

In [ ]:
total_pairs = [("MDP(S)", "GMV(S)"), ("MDP(LW)", "MDP(S)")]
beta_pairs = [("MDP(S)", "EW"), ("MDP(S)", "IV")]

fig, axes = plt.subplots(2, 1, figsize=(9.5, 9), sharex=True)
ml.plotting.cumulative_paired(
    paired_series, total_pairs, split_ts, ax=axes[0],
    title="(a) H3 MDP(S)-GMV(S) and H5 MDP(LW)-MDP(S): total", ylabel="cumulative diff (%)", legend_kw={"fontsize": 8},
)
ml.plotting.cumulative_paired(
    paired_series, beta_pairs, split_ts,
    beta={f"{a} - {b}": diff_capm_table.loc[f"{a} - {b}", "beta_diff"] for a, b in beta_pairs},
    mkt_excess=MKT_simple, ax=axes[1],
    title="(b) H4 MDP(S)-EW and H5 MDP(S)-IV: solid = beta-adjusted, dashed = total",
)
plt.tight_layout(); plt.show()

The endpoints match 17.2 years × the annualized means: MDP(S) − GMV(S) ends near +11% and MDP(LW) − MDP(S) near +3.5%; MDP(S) − EW ends near −55% total but +18% β-adjusted, MDP(S) − IV near −24% and +10%. Most of MDP's gain over GMV comes after 2015, with a jump from September 2025 to February 2026. MDP − EW's total line falls mostly in 2009–2012 and 2020–21, when equities rallied.

### 7.5 Sensitivity: MDP(LW) and IV

½L1 weight distance from MDP(S) at each rebalance, and MDP(LW)'s effective N.

In [ ]:
wlog_mdp_s = results["MDP(S)"]["wlog"]
wlog_mdp_lw = results["MDP(LW)"]["wlog"]
wlog_iv = results["IV"]["wlog"]

halfL1_lw_vs_s = 0.5 * (wlog_mdp_lw - wlog_mdp_s).abs().sum(axis=1)
halfL1_mdp_vs_iv = 0.5 * (wlog_mdp_s - wlog_iv).abs().sum(axis=1)
effN_mdp_lw_ts = 1.0 / (wlog_mdp_lw ** 2).sum(axis=1)

halfL1_shown = pd.DataFrame(
    {"median ½L1 (%)": [halfL1_lw_vs_s.median(), halfL1_mdp_vs_iv.median()],
     "max ½L1 (%)": [halfL1_lw_vs_s.max(), halfL1_mdp_vs_iv.max()]},
    index=["MDP(LW) vs MDP(S)", "MDP(S) vs IV"])
print(f"median effN, MDP(LW): {float(effN_mdp_lw_ts.median()):.2f}")
ml.reporting.show_table(halfL1_shown, pct=list(halfL1_shown.columns), pct_dp=2)

MDP(LW) moves weights by a median 8.99% (max 26.49%) from MDP(S), with median effN 4.04; IV sits a median 61.22% away. LW is a small perturbation of MDP; IV is a different portfolio.

## Verdict table

The §2 rules applied mechanically to the statistics in §7.2–§7.4; `[applied: …]` states how an ambiguous rule was read.

In [ ]:
reg = ml.inference.load_registration("nb05")

h2_uup_argmax = results["MDP(S)"]["wlog"].loc[asof_2022].idxmax() == "UUP"
h2_holds_2022 = (uup_2022_mdp >= uup_2022_gmv) and h2_uup_argmax
d4 = decomp["MDP(S) - EW"]
t_full_gmv = float(paired_table.loc[("MDP(S) - GMV(S)", "full"), "t"])
t_test_gmv = float(paired_table.loc[("MDP(S) - GMV(S)", "test (> split)"), "t"])
t_full_ew = float(paired_table.loc[("MDP(S) - EW", "full"), "t"])
mean_test_ew = float(paired_table.loc[("MDP(S) - EW", "test (> split)"), "ann_mean_diff"])
t_alpha_ew = float(diff_capm_table.loc["MDP(S) - EW", "t_alpha_diff"])
t_full_lw = float(paired_table.loc[("MDP(LW) - MDP(S)", "full"), "t"])
t_test_lw = float(paired_table.loc[("MDP(LW) - MDP(S)", "test (> split)"), "t"])
t_alpha_iv = float(diff_capm_table.loc["MDP(S) - IV", "t_alpha_diff"])

stats = {
    "H1": dict(frac_full=le_full / n_full, frac_test=le_test / n_test,
               display=f"full {le_full}/{n_full}, test {le_test}/{n_test}"),
    "H2": dict(holds_2022=h2_holds_2022, frac_full=ge_full / n_full2, frac_test=ge_test / n_test2,
               display=f"full {ge_full}/{n_full2}, test {ge_test}/{n_test2}, "
                       f"2022: MDP≥GMV={uup_2022_mdp >= uup_2022_gmv}, UUP largest MDP weight={h2_uup_argmax}"),
    "H3": dict(t_full=t_full_gmv, t_test=t_test_gmv, display=f"t_full={t_full_gmv:.2f}, t_test={t_test_gmv:.2f}"),
    "H4": dict(t_full=t_full_ew, mean_test=mean_test_ew, beta_part=d4["beta_part"], d_alpha=d4["d_alpha"],
               t_alpha=t_alpha_ew,
               display=f"t_full={t_full_ew:.2f}, mean_test={100 * mean_test_ew:.2f}%, "
                       f"|β part|={100 * abs(d4['beta_part']):.2f}% vs |Δα|={100 * abs(d4['d_alpha']):.2f}%, "
                       f"t(Δα)={t_alpha_ew:.2f}"),
    "H5_LW": dict(t_full=t_full_lw, t_test=t_test_lw, display=f"t_full={t_full_lw:.2f}, t_test={t_test_lw:.2f}"),
    "H5_IV": dict(t=t_alpha_iv, display=f"t={t_alpha_iv:.2f}"),
}

vt = ml.inference.evaluate(reg, stats)

with pd.option_context("display.max_colwidth", None):
    display(vt.set_index("item"))

## §8 TAKEAWAY — the UUP corner comes from correlation, not volatility

- **MDP keeps the UUP corner (H2 supported).** UUP weight MDP ≥ GMV at 187 of 208 rebalances (38 of 40 test); at 2022-12-31 59.22% vs 58.12%. Correlation makes UUP the largest MDP risk weight (0.4167; mean correlation −0.47); its low volatility lifts the capital weight above that.
- **MDP is more concentrated than GMV (H1 supported).** effN(MDP) ≤ effN(GMV) at 146 of 208 (34 of 40 test); median 3.35 vs 3.79.
- **MDP − GMV: the null holds, narrowly (H3).** +0.64%/yr, t 1.98 full and 1.68 test.
- **MDP trails EW through β (H4 supported, at the threshold).** −3.17%/yr, t −2.00; β part −4.19%/yr, Δα +1.02%/yr (t 1.00). The H5 nulls hold: MDP(LW) − MDP(S) t 1.31, MDP(S) − IV Δα t 0.71.
- **Against the benchmarks:** full-window Sharpe 0.76 vs EW 0.74 and 60/40 0.91; test 0.98 vs 0.91 and 1.00. Plain-OLS t throughout; notebook 08 tests each run's Sharpe ratio and CAPM α with multiple-testing control.

Scoring decorrelation alone keeps the corner; notebook 06 equalizes risk contributions instead.